# 리포트 10 — 무엇을 조명원으로 쓸 수 있나

> LTE·5G·WiFi 가 각각 **얼마나 자주, 얼마나 넓게** 신호를 내주는가. 5G 는 대역이 넓은 대신 상시 신호가 드물어 **이중고**가 된다.

이 권은 아래 절로 이루어진다. 각 절은 **한 일 · 결과 · 방법 · 재현** 을 자기 앞에 달고 있어, 필요한 절만 따로 읽어도 된다.

| 절 | 무엇을 말하나 | 만든 곳 |
|---|---|---|
| 1 | 상시이면서 내용을 미리 아는 신호는 표준마다 하나씩 있다 | `_parts/44_illuminators.ipynb` |
| 2 | 5G 는 좁고 드물다 — 두 배의 대가를 치른다 | `_parts/45_5g-double-cost.ipynb` |
| 3 ⭐ | 여섯 항목은 닫힌형이고, 점유 대가만 몬테카를로 격자에서 읽는다 | `_parts/46_cost-ledger.ipynb` |
| 4 | 바이스태틱 거리 분해능은 c/B, 잡음대역은 √(B/fs) 로 고정한다 | `_parts/47_range-convention.ipynb` |
| 5 | 같은 자원격자를 독립 변조기에 넣어 상관 1.0000 을 얻었다 | `_parts/48_waveform-check.ipynb` |
| 6 | 검출기가 실제로 쓰는 커널 그대로 모호함수를 그렸다 | `_parts/49_ambiguity.ipynb` |
| 7 | 5G SSB 는 걷는 드론에서 접힌다 | `_parts/50_doppler-fold.ipynb` |

⭐ 표시한 절 하나만 읽어도 이 권의 결론은 선다.

숫자는 전부 계산 결과 JSON(원장)에서 주입된다 — 절 끝 «출처» 표가 그 파일과 키다. 원장이 다시 계산되면 빌더를 돌리는 것만으로 본문 숫자가 따라 바뀐다.

전체 목차는 [reports/README.md](README.md) 이고, 열다섯 권의 지도는 [리포트 1 «이 연구가 묻는 것과 답한 방식»](01_map.ipynb) 다.


---

## 절 1. 상시이면서 내용을 미리 아는 신호는 표준마다 하나씩 있다



> ### 한 일
> **WiFi · LTE · 5G NR 세 표준의 자원격자를 규격서대로 세우고, 패시브가 상관에 걸 수 있는 상시 기준신호를 표준마다 하나씩 골라 제원을 격자에서 직접 쟀다.**

### 결과
1. 두 조건(내용을 미리 안다 · 아무 셀이나 늘 켠다)을 함께 만족하는 신호는 표준마다 하나다 — WiFi VHT-LTF($B_{ref}$ 76.56 MHz [^1]) · LTE CRS(17.98 MHz [^2]) · 5G SSB(7.20 MHz [^3]).
2. 거리 눈금 $\Delta R_b = c/B_{ref}$ 는 3.9 [^4] · 16.7 [^5] · 41.6 m [^6] 로 세 표준이 한 자릿수 배 이상 갈린다.
3. $B_{ref}$ 는 기준신호가 차지한 부반송파의 양끝 span 이라 안쪽 널 톤을 포함한다 — WiFi 는 span 76.562 MHz [^1] 가 채널 점유대역 75.625 MHz [^7] 보다 넓다.
4. 프레임 안에서 기준신호가 실제로 차지하는 몫은 9.13% [^8] (WiFi) · 3.22% [^9] (LTE) · 1.45% [^10] (5G) 다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 자원격자 | `TS 36.211`(CRS) · `TS 38.211`(SSB) · `IEEE 802.11ac`(VHT-LTF) 를 읽어 세웠다 — `src/waveforms.py:258`(WiFi) · `:313`(LTE) · `:370`(5G) |
| 제원 측정 | 선언값을 옮겨 적지 않고 **생성한 격자에서 직접 쟀다** — $B_{ref}$ 는 `src/waveforms.py:237`, $\Delta R_b$ 는 `:144` |
| 거리 규약 | 바이스태틱 거리합 $R_b = R_1 + R_2 - L$ 이라 분해능은 $c/B_{ref}$ 다 — 모노스태틱 교과서 값의 두 배다 |

### 재현

```bash
cd /home/yunjung/workspace/sionna2
~/.venvs/py312/bin/python src/viz_report2.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part08_illuminators.py
```

| | |
|---|---|
| 출력 | `outputs/report2_waveform_rcs.json`, `outputs/report03_illuminators.json` |
| 소요 | ① 3412 s [^11] (대부분 같은 스크립트의 RCS 스윕) · ② CPU 20초 안쪽 |

---


## 패시브가 상관을 걸 수 있는 신호는 어떤 것인가

패시브 수신기는 남이 쏘는 신호를 빌려 쓴다. 그 신호에 상관을 걸려면 두 조건이 **동시에** 서야 한다.

**① 내용을 미리 안다.** 데이터는 매 순간 바뀌므로 규격이 고정한 기준신호가 그 자리를 맡는다.

**② 아무 셀이나 늘 켠다.** 상시 신호라야 표적이 지나가는 그 순간에도 공중에 있다.

두 조건을 다 만족하는 신호는 표준마다 **하나씩**이다.


## 격자에서 잰 제원

| 표준 | 상시 기준신호 | 반송파 | 채널 점유대역 | $B_{ref}$ | $\Delta R_b=c/B_{ref}$ |
|---|---|---|---|---|---|
| WiFi 802.11ac | VHT-LTF | 5.21 GHz [^12] | 75.6 MHz [^7] | 76.6 MHz [^1] | 3.9 m [^4] |
| LTE Rel-9 | CRS | 1.843 GHz [^13] | 18.0 MHz [^14] | 18.0 MHz [^2] | 16.7 m [^5] |
| 5G NR Rel-16 | SSB | 3.50 GHz [^15] | 98.3 MHz [^16] | 7.2 MHz [^3] | 41.6 m [^6] |


## $B_{ref}$ 는 span 이다

$B_{ref}$ 는 기준신호가 차지한 부반송파의 **양끝 span** 이다(`src/waveforms.py:237`). 안쪽 널 톤이 그 안에 들어오므로 WiFi 는 span 이 점유대역보다 넓게 나온다.

이 정의가 거리 눈금을 정한다. 채널 대역이 아니라 **상관에 쓰는 대역**이 분해능을 만들기 때문이다 — 5G 채널은 98.3 MHz [^16] 인데 상시 SSB 체제의 거리 눈금은 41.6 m [^6] 다. 채널 대역이 다 열린 체제의 값 3.05 m [^17] 는 **절 3** «여섯 항목은 닫힌형이고» 가 낙관적 상한으로 함께 싣는다.

규약의 정확한 형태는 **절 4** «바이스태틱 거리 분해능은 c/B, 잡음대역은 √(B/fs) 로 고정한다» 가 든다.


![report03_f1_grid](../outputs/figures/report03_f1_grid.png)

**그림 1.** 유휴 셀이 실제로 켜는 칸은 어디이고, 그중 패시브가 상관에 쓰는 것은 무엇인가?


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| `src/waveforms.py:112` 의 `PILOT_RATE_HZ` 를 트래픽 시나리오 파라미터로 올린다 | WiFi PRF 가 유휴 AP ~ 혼잡 AP 범위로 확정되고 이 편의 제원표가 시나리오별로 선다 | `src/waveforms.py:112` |
| X410 으로 실제 셀을 캡처해 격자 좌표를 대조한다 | CRS · SSB · VHT-LTF 의 자원요소 좌표가 실측으로 확정된다 | [리포트 14 절 4 «X410 의 12-bit ADC 동적범위가 직…»](14_robustness.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 17개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/report2_waveform_rcs.json` | `reference.G1.wifi.ref_bw_mhz` | 76.56 |
| [^2] | `outputs/report2_waveform_rcs.json` | `reference.G1.lte.ref_bw_mhz` | 17.98 |
| [^3] | `outputs/report2_waveform_rcs.json` | `reference.G1.nr.ref_bw_mhz` | 7.2 |
| [^4] | `outputs/report2_waveform_rcs.json` | `reference.G1.wifi.dR_m` | 3.916 |
| [^5] | `outputs/report2_waveform_rcs.json` | `reference.G1.lte.dR_m` | 16.67 |
| [^6] | `outputs/report2_waveform_rcs.json` | `reference.G1.nr.dR_m` | 41.64 |
| [^7] | `outputs/report2_waveform_rcs.json` | `reference.G1.wifi.chan_bw_mhz` | 75.62 |
| [^8] | `outputs/report2_waveform_rcs.json` | `reference.G1.wifi.occ_pct` | 9.135 |
| [^9] | `outputs/report2_waveform_rcs.json` | `reference.G1.lte.occ_pct` | 3.223 |
| [^10] | `outputs/report2_waveform_rcs.json` | `reference.G1.nr.occ_pct` | 1.447 |
| [^11] | `outputs/report2_waveform_rcs.json` | `meta.runtime_s` | 3412 |
| [^12] | `outputs/report2_waveform_rcs.json` | `reference.G1.wifi.carrier_ghz` | 5.21 |
| [^13] | `outputs/report2_waveform_rcs.json` | `reference.G1.lte.carrier_ghz` | 1.843 |
| [^14] | `outputs/report2_waveform_rcs.json` | `reference.G1.lte.chan_bw_mhz` | 18 |
| [^15] | `outputs/report2_waveform_rcs.json` | `reference.G1.nr.carrier_ghz` | 3.5 |
| [^16] | `outputs/report2_waveform_rcs.json` | `reference.G1.nr.chan_bw_mhz` | 98.28 |
| [^17] | `outputs/report2_waveform_rcs.json` | `reference.G1.nr.chan_dR_m` | 3.05 |


---

## 절 2. 5G 는 좁고 드물다 — 두 배의 대가를 치른다



> ### 한 일
> **상시 기준신호 체제에서 LTE CRS 와 5G SSB 를 거리 축과 속도 축 두 곳에서 나란히 재고, 두 축의 격차를 배수로 적었다.**

### 결과
1. 거리 축 — SSB 는 $B_{ref}$ 가 좁아 $\Delta R_b$ 가 41.6 m [^18] 로 LTE CRS 16.7 m [^19] 의 2.5 [^20]배로 거칠다.
2. 속도 축 — SSB 의 물리 반복률이 50 Hz [^21] 라 무모호 속도가 1.07 m/s [^22] 다. LTE CRS 는 40.7 m/s [^23] 로 38 [^24]배 넓다.
3. 두 축 위에 반송파 항이 하나 더 얹힌다 — LTE→5G 의 $\lambda^2$ 는 -5.57 dB [^25] 다.
4. PRS 는 측위 세션이 설정될 때 켜지는 옵션이고, 남의 셀을 빌리는 수신기의 기본선은 상시 SSB 다 — PRS 를 켠 수치는 낙관적 상한으로 읽는다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 두 축의 정의 | 거리 축은 $B_{ref}$ 가, 속도 축은 물리 반복률이 정한다 — 둘 다 규격이 고정한 자원격자에서 나오는 닫힌형이다 |
| 비교 체제 | 상시 기준신호 체제(G1)에서 잰다. PRS 체제(G2·G3)는 같은 그림에 함께 싣고 낙관적 상한으로 읽는다 |
| $\lambda^2$ 항 | EIRP 고정 · 수신 안테나 **이득** 고정 전제에서 선다 — `src/freespace_link.py:371` |

### 재현

```bash
cd /home/yunjung/workspace/sionna2
~/.venvs/py312/bin/python src/viz_report2.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part08_illuminators.py
```

| | |
|---|---|
| 출력 | `outputs/report2_waveform_rcs.json`, `outputs/report03_illuminators.json` |
| 소요 | CPU 20초 안쪽 (JSON 을 읽어 노트북을 조립한다) |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| **절 1** «상시이면서 내용을 미리 아는 신호는 표준마다…» | 세 표준의 상시 기준신호와 그 제원 |

---


## 세대가 최신일수록 조명원으로 유리하다는 통념

이 통념을 세 항목이 뒤집는다. 5G 는 채널이 넓지만 **상시로 켜는 부분**은 좁고, 그 부분이 다시 오는 간격은 길다. 패시브가 빌려 쓰는 것은 채널이 아니라 그 좁은 상시 부분이다.

**PRS 는 측위 세션이 설정될 때 켜지는 옵션**이고, 남의 셀을 빌리는 패시브 수신기의 기본선은 상시 신호인 **SSB** 다.


## 두 축에서 각각 얼마인가

| 축 | 정하는 것 | LTE CRS | 5G SSB | 격차 |
|---|---|---|---|---|
| 거리 $\Delta R_b$ | $B_{ref}$ | 16.7 m [^19] | 41.6 m [^18] | 2.5 [^20]배 거칢 |
| 속도 $v_{max}$ (물리 PRF) | PRF | 40.7 m/s [^23] | 1.07 m/s [^22] | 38 [^24]배 넓음 |

여기에 반송파가 $\lambda^2$ -5.57 dB [^25] 를 더한다 — 그 항의 성립 조건과 크기는 **절 3** «여섯 항목은 닫힌형이고» 가 든다.


![report03_f2_reference](../outputs/figures/report03_f2_reference.png)

**그림 1.** 기준신호의 넓이와 반복이 거리·속도 눈금을 각각 얼마로 정하는가?


## 속도 축의 대가는 접힘으로 나타난다

SSB 의 반복률 50 Hz [^21] 는 걷는 속도의 드론에서도 도플러를 접는다. 그 접힘을 실제 커널에서 잰 값이 **절 7** «5G SSB 는 걷는 드론에서 접힌다» 에 있다.

거리 축의 대가는 조명원 선택의 dB 원장에 다른 항목과 함께 들어간다 — **절 3** «여섯 항목은 닫힌형이고».


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| PRS 를 켠 체제에서 같은 두 축을 다시 잰다 | 낙관적 상한과 상시 기준선이 거리·속도 축에서 각각 몇 배 갈리는지 확정된다 | `src/waveforms.py:370` → **절 3** «여섯 항목은 닫힌형이고» |
| 실측 설계에서 수신 안테나를 확정하고 $\lambda^2$ 항의 전제를 다시 잰다 | $\lambda^2$ -9.03 dB [^26] 의 부호가 실제 안테나에서 확정된다 | `src/freespace_link.py:371` → [리포트 14 절 4 «X410 의 12-bit ADC 동적범위가 직…»](14_robustness.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 9개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^18] | `outputs/report2_waveform_rcs.json` | `reference.G1.nr.dR_m` | 41.64 |
| [^19] | `outputs/report2_waveform_rcs.json` | `reference.G1.lte.dR_m` | 16.67 |
| [^20] | `outputs/report03_illuminators.json` | `ratios.drb_nr_over_lte` | 2.498 |
| [^21] | `outputs/report2_waveform_rcs.json` | `reference.G1.nr.prf_hz` | 50 |
| [^22] | `outputs/report2_waveform_rcs.json` | `reference.G1.nr.vmax_ms` | 1.071 |
| [^23] | `outputs/report2_waveform_rcs.json` | `reference.G1.lte.vmax_ms` | 40.67 |
| [^24] | `outputs/report03_illuminators.json` | `ratios.vmax_lte_over_nr` | 37.98 |
| [^25] | `outputs/report03_illuminators.json` | `lambda2.lte_to_nr_db` | -5.571 |
| [^26] | `outputs/report03_illuminators.json` | `lambda2.span_db` | -9.026 |


---

## 절 3. 여섯 항목은 닫힌형이고, 점유 대가만 몬테카를로 격자에서 읽는다



> ### 한 일
> **조명원 선택이 무는 dB 격차를 항목별로 모아 원장을 만들고, 항목마다 그 값을 닫는 방식이 무엇인지를 함께 적었다.**

### 결과
1. 원장 항목은 전부 **같은 표적·같은 기하에서 잰 두 양의 비**라 표적 σ 가 분자와 분모에서 상쇄된다 — 반송파 $\lambda^2$ -9.03 dB [^27] (밴드 양끝) · WiFi 패킷 듀티 -12.84 dB [^28] · CPI 규약 3.01 dB [^29].
2. 여섯 항목은 자원격자 · 반송파 · 관측시간에서 **닫힌형**으로 나온다.
3. 점유 대가만 검출 몬테카를로의 EIRP 격자에서 **읽은** 값이다 — 격자점 차 18 dB [^30], 참값 구간 12 [^31]~24 dB [^32], $P_d$ 선형보간 16.4 dB [^33].
4. 그 격자는 눈금 6 dB [^34] · 시행 60 [^35]회 · 표적 mavic4pro [^36] · radial [^37] 한 점에서 읽었다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 항목의 형태 | 전부 두 양의 비다 — $\lambda^2$ 는 반송파 비, 듀티는 시간 비, 기준신호 에너지는 자원격자 위의 에너지 비 |
| 점유 대가 | 같은 표적·같은 기하에서 $P_d$ 0.5 [^38] 를 넘기는 EIRP 차를 6 dB [^34] 격자에서 읽는다 |
| 부호 규약 | 표의 부호는 원본 JSON 그대로이고, 그림은 **음수 = 손해**로 부호를 맞춰 다시 그린 것이다 |
| 점유 대가 안에 든 것 | G1→G3 은 점유율과 함께 기준신호 대역도 7.2 MHz [^39] → 98.28 MHz [^40] 로 넓어진 값이다 — 두 항을 가르는 대역고정 스윕은 다음 단계에 있다 |

### 재현

```bash
cd /home/yunjung/workspace/sionna2
~/.venvs/py312/bin/python src/viz_report2.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/verify_ambiguity.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/report4_fixups.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part08_illuminators.py
```

| | |
|---|---|
| 출력 | `outputs/report2_waveform_rcs.json`, `outputs/report4_fixups.json`, `outputs/report5_results.json`, `outputs/report03_illuminators.json` |
| 소요 | ③ 556 s [^41] · ④ CPU 20초 안쪽 |
| 비고 | `outputs/report5_results.json` 는 검출 몬테카를로가 이미 남긴 것이다 — 이 편은 그중 `A_occupancy` 만 인용한다(재실행 불필요). |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| **절 2** «5G 는 좁고 드물다» | 5G 가 거리·속도 두 축에서 무는 대가 |

---


## 원장 — 무엇이 각 항목의 유효숫자를 정하나

조명원 선택이 만드는 dB 격차를 한 표에 모은다. 오른쪽 열이 그 항목을 **닫는 방식**이다.


| 항목 | 값 | 무엇의 비인가 | 닫는 방식 |
|---|---|---|---|
| 점유 대가 (5G · 상시 vs 풀로드) | 18 dB [^30] | 같은 표적·같은 기하에서 $P_d$ 0.5 [^38] 를 넘기는 EIRP 차 | 몬테카를로 격자 읽기 |
| 기준신호 에너지 격차 (같은 쌍) | 12.70 dB [^42] | $E_{ref}$(G3) / $E_{ref}$(G1) — 상관에 쓰는 에너지만 | 닫힌형 — 자원격자 |
| 반송파 $\lambda^2$ (LTE→WiFi) | -9.03 dB [^43] | $20\log_{10}(\lambda/\lambda_{ref})$ — EIRP·수신이득 고정 | 닫힌형 — 반송파 |
| 반송파 $\lambda^2$ (LTE→5G) | -5.57 dB [^44] | 위와 같음 | 닫힌형 — 반송파 |
| WiFi 파일럿 / 총 송신 에너지 | -11.27 dB [^45] | G3 격자에서 상관에 쓰는 몫 | 닫힌형 — 자원격자 |
| WiFi 패킷 듀티 | -12.84 dB [^28] | 패킷이 공중에 있는 시간 비율 | 닫힌형 — 시간 |
| CPI 규약 격차 | 3.01 dB [^29] | 같은 프레임 수 M 이 5G 에 주는 관측시간이 절반 | 닫힌형 — 관측시간 |


## 각 항목이 서는 조건

| 항목 | 성립 조건 | 크기 |
|---|---|---|
| 반송파 $\lambda^2$ | EIRP 고정 · 수신 안테나 **이득** 고정 (`src/freespace_link.py:371`) | 수신 **개구면적**을 고정하면 부호가 뒤집힌다 |
| CPI 규약 | 같은 M 이 5G 에 주는 관측시간이 절반 | 3.01 dB [^29] — 뒤 편들은 관측시간을 맞춘 뒤 비교한다 |
| WiFi 두 항목 | 에너지 비 · 시간 비 — 서로 다른 양이다 | -11.27 dB [^45] · -12.84 dB [^28] |


## 점유 대가는 왜 격자에서 읽는가

이 항목만 닫힌형이 없다. 같은 표적·같은 기하에서 $P_d$ 0.5 [^38] 를 넘기는 EIRP 를 상시 체제와 풀로드 체제에서 각각 찾아 그 차를 읽는다. 격자 눈금이 유효숫자를 정한다.

| 무엇 | 값 |
|---|---|
| 격자점 차 | 18 dB [^30] |
| 참값 구간 | 12 [^31]~24 dB [^32] |
| $P_d$ 선형보간 | 16.4 dB [^33] |
| 격자 눈금 · 시행 | 6 dB [^34] · 60 [^35]회 |
| 기준신호 대역 (G1 → G3) | 7.2 MHz [^39] → 98.28 MHz [^40] |


![report03_f3_occupancy](../outputs/figures/report03_f3_occupancy.png)

**그림 1.** 셀이 데이터로 바빠지면 패시브의 거리분해능도 같이 좋아지는가?


![report03_f4_ledger](../outputs/figures/report03_f4_ledger.png)

**그림 2.** 조명원 선택이 무는 대가는 항목별로 몇 dB 인가?


## 이 원장이 σ 논의와 독립인 이유

항목마다 분자와 분모가 같은 표적·같은 기하에서 나온다. 그래서 표적 σ 의 절대레벨이 X dB 움직여도 세 조명원의 **순위와 격차는 그대로**이고, 움직이는 것은 절대 검출거리뿐이다.

검출 결과 편들이 여기에 σ 와 기하를 곱해 절대 거리를 낸다 — [리포트 12 절 3 «세 밴드에서 값이 다른 항은 λ² 와 σ 둘뿐이다»](12_observability.ipynb).


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| EIRP 격자를 6 dB [^34] 에서 2 dB 로 좁히고 기준신호 대역을 고정한 점유 스윕을 돌린다 | 18.0 dB [^30] 안에서 점유 항과 대역 항의 크기가 갈린다 | `benchmark/run_matrix.py:300` |
| 표적 mavic4pro [^36] · 시나리오 radial [^37] 한 점에서 읽은 점유 대가를 기체·기하로 넓힌다 | 점유 대가가 표적·기하에 얼마나 의존하는지가 수치로 확정된다 | [리포트 13 절 2 «앵커 σ 위의 R90 은 비교가능 12칸에서…»](13_results.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 19개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^27] | `outputs/report03_illuminators.json` | `lambda2.span_db` | -9.026 |
| [^28] | `outputs/report4_fixups.json` | `F4_linkbudget.wifi_pilot_fraction.packet_duty_db` | -12.84 |
| [^29] | `outputs/report4_fixups.json` | `F4_linkbudget.cpi_asymmetry.span_db` | 3.01 |
| [^30] | `outputs/report03_illuminators.json` | `occupancy_cost.value_db` | 18 |
| [^31] | `outputs/report03_illuminators.json` | `occupancy_cost.bracket_lo_db` | 12 |
| [^32] | `outputs/report03_illuminators.json` | `occupancy_cost.bracket_hi_db` | 24 |
| [^33] | `outputs/report03_illuminators.json` | `occupancy_cost.interp_db` | 16.41 |
| [^34] | `outputs/report03_illuminators.json` | `occupancy_cost.eirp_grid_step_db` | 6 |
| [^35] | `outputs/report03_illuminators.json` | `occupancy_cost.n_trials` | 60 |
| [^36] | `outputs/report03_illuminators.json` | `occupancy_cost.drone` | mavic4pro |
| [^37] | `outputs/report03_illuminators.json` | `occupancy_cost.scen` | radial |
| [^38] | `outputs/report03_illuminators.json` | `occupancy_cost.pd_threshold` | 0.5 |
| [^39] | `outputs/report03_illuminators.json` | `occupancy_cost.ref_bw_G1_mhz` | 7.2 |
| [^40] | `outputs/report03_illuminators.json` | `occupancy_cost.ref_bw_G3_mhz` | 98.28 |
| [^41] | `outputs/report4_fixups.json` | `_meta.runtime_s` | 555.5 |
| [^42] | `outputs/report03_illuminators.json` | `ref_energy_gap_G1_to_G3_db.nr` | 12.7 |
| [^43] | `outputs/report03_illuminators.json` | `lambda2.lte_to_wifi_db` | -9.026 |
| [^44] | `outputs/report03_illuminators.json` | `lambda2.lte_to_nr_db` | -5.571 |
| [^45] | `outputs/report4_fixups.json` | `F4_linkbudget.wifi_pilot_fraction.pilot_over_tx_energy_db` | -11.27 |


---

## 절 4. 바이스태틱 거리 분해능은 c/B, 잡음대역은 √(B/fs) 로 고정한다



> ### 한 일
> **이 프로젝트가 쓰는 거리 분해능과 잡음대역 정규화의 정의를 하나로 고정하고, 모노스태틱 교과서 값과의 배수를 표에 적었다.**

### 결과
1. 거리축은 바이스태틱 거리합 $R_b = R_1 + R_2 - L$ 이라 분해능이 $\Delta R_b = c/B_{ref}$ 다. 모노스태틱 교과서 값 $c/2B$ 는 그 절반이고, 비는 2 [^46]배다.
2. 세 파형의 값은 3.92 [^47] · 16.67 [^48] · 41.64 m [^49] 이고, 모노 등가는 각각 1.96 [^50] · 8.33 [^51] · 20.82 m [^52] 다.
3. 잡음대역 정규화는 $\sqrt{B/f_s}$ 로 고정한다 — 세 파형의 $B/f_s$ 는 0.9453 [^53] · 0.5859 [^54] · 0.7998 [^55] 다.
4. 두 규약을 섞으면 분해능을 그 배수만큼 낙관하게 된다 — 그래서 한 곳에서 정의하고 모든 편이 그 정의를 인용한다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 거리 규약 | $R_b = c\tau$ 에 계수 2 가 없다. 그래서 $\Delta R_b = c/B_{ref}$ 이고, 모노스태틱 $c/2B$ 는 그 절반이다 |
| 잡음대역 정규화 | 선언 대역 $B$ 와 표본율 $f_s$ 가 다르면 주입 진폭을 $\sqrt{B/f_s}$ 만큼 낮춘다 — `benchmark/run_min_cell.py:131` |
| 행 순서 검사 | $B/f_s$ 행이 (WiFi, LTE, 5G) 순서인지 격자에서 다시 계산해 대조한다. 행이 뒤섞이면 빌드가 멈춘다 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/verify_ambiguity.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/report4_fixups.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part08_illuminators.py
```

| | |
|---|---|
| 출력 | `outputs/verify_ambiguity.json`, `outputs/report4_fixups.json` |
| 소요 | ② GPU 1장 수 분 · ③ 556 s [^56] |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| **절 1** «상시이면서 내용을 미리 아는 신호는 표준마다…» | $B_{ref}$ 가 어떻게 정의되는가 |

---


## 거리 규약 — 바이스태틱 $c/B$

송신기와 수신기가 다른 자리에 있으면 표적이 만드는 지연은 두 경로의 합에서 직접경로를 뺀 값이다. 그 축에서 분해능은 $c/B_{ref}$ 이고, 모노스태틱 교과서 값 $c/2B$ 는 그 절반이다.

| 파형 | $B/f_s$ | $\Delta R_b = c/B_{ref}$ | 모노 등가 $c/2B$ |
|---|---|---|---|
| WiFi 80MHz | 0.9453 [^53] | 3.92 m [^47] | 1.96 m [^50] |
| LTE 20MHz | 0.5859 [^54] | 16.67 m [^48] | 8.33 m [^51] |
| 5G 100MHz | 0.7998 [^55] | 41.64 m [^49] | 20.82 m [^52] |


## 분해능과 격자 간격은 다른 양이다

$\Delta R_b$ 는 **선언 규약**이고, 표본율이 정하는 거리 빈 $c/f_s$ 는 격자 간격이다. 두 값을 한 표에 병기해 섞이지 않게 둔다.

검출기 쪽에서 같은 규약을 쓰고 같은 값을 싣는 표가 [리포트 12 절 1 «한 순간의 (R_b, f_d) 는 랭크 2 이고, 수신기를 하나 더하면 위치가 풀린다»](12_observability.ipynb) 에 있다.


## 잡음대역 정규화 — $\sqrt{B/f_s}$

선언 대역 $B$ 와 표본율 $f_s$ 가 다르면 표본당 잡음이 그만큼 달라진다. 주입 진폭을 $\sqrt{B/f_s}$ 로 낮춰야 매치드필터 출력 SNR 이 파형 간 공정해진다(`benchmark/run_min_cell.py:131`).

이 규약이 있어야 대역이 다른 세 파형을 하나의 SNR 축에 올릴 수 있다. 그 축 위에서 세 파형을 비교한 결과가 [리포트 12 절 4 «자유공간 형상에서 문턱을 다시 재니 세 밴드가 SNR90 하나를 공유한다»](12_observability.ipynb) 다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 두 거리 규약이 섞이지 않았는지 검출기 설정 검사에 한 줄로 넣는다 | 규약 혼용이 조용히 지나가지 않고 빌드에서 멈춘다 | `src/passive_process.py:383` |
| $\sqrt{B/f_s}$ 규약을 X410 캡처 경로에도 건다 | 실측 SNR 축이 시뮬 축과 같은 정규화 위에 선다 | `src/experiment_x410.py` → [리포트 14 절 4 «X410 의 12-bit ADC 동적범위가 직…»](14_robustness.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 11개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^46] | `outputs/report4_fixups.json` | `F3_ambiguity.resolution_convention_conflict.rows[0].factor` | 2 |
| [^47] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.dR_theory_m` | 3.916 |
| [^48] | `outputs/verify_ambiguity.json` | `waveforms.lte_G1.dR_theory_m` | 16.67 |
| [^49] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.dR_theory_m` | 41.64 |
| [^50] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.dR_mono_theory_m` | 1.958 |
| [^51] | `outputs/verify_ambiguity.json` | `waveforms.lte_G1.dR_mono_theory_m` | 8.335 |
| [^52] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.dR_mono_theory_m` | 20.82 |
| [^53] | `outputs/report4_fixups.json` | `F4_linkbudget.straddle.rows[0].b_over_fs` | 0.9453 |
| [^54] | `outputs/report4_fixups.json` | `F4_linkbudget.straddle.rows[1].b_over_fs` | 0.5859 |
| [^55] | `outputs/report4_fixups.json` | `F4_linkbudget.straddle.rows[2].b_over_fs` | 0.7998 |
| [^56] | `outputs/report4_fixups.json` | `_meta.runtime_s` | 555.5 |


---

## 절 5. 같은 자원격자를 독립 변조기에 넣어 상관 1.0000 을 얻었다



> ### 한 일
> **우리 변조기와 Sionna PHY 의 `OFDMModulator` 에 같은 자원격자를 넣고 두 시간파형의 상관과 NMSE 를 재, 변조 단계를 독립 구현으로 채점했다.**

### 결과
1. 세 파형 모두 상관이 소수 넷째 자리까지 1 이다 — WiFi 1.0000 [^57] · LTE 1.0000 [^58] · 5G 1.0000 [^59].
2. NMSE 는 WiFi -138.3 [^60] · LTE -135.6 [^61] · 5G -135.2 dB [^62] 로 float32 반올림 바닥에 붙는다.
3. 대조의 **분해력**을 같은 표에 싣는다 — 심볼별 CP 배열 대신 첫 CP 스칼라만 넘기면 LTE 상관이 0.06 [^63], 5G 가 0.05 [^64] 로 무너진다. CP 가 심볼마다 같은 WiFi 는 1.0000 [^65] 로 남는다.
4. 대조는 G3(풀로드) 격자에서 돈다 — 표본 수 61440 [^66] · $f_s$ 122.88 MHz [^67] (5G 기준).


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 무엇을 채점하나 | 격자를 신호로 바꾸는 **변조 단계**다 — IFFT 규약(fftshift 방향·정규화), CP 복사, 심볼별 이어붙이기 순서 |
| 상대 구현 | `sionna.phy.ofdm.OFDMModulator`(Sionna 2.0.1) — 우리 코드를 한 줄도 공유하지 않는다 |
| 분해력 시험 | 재변조 쪽에 심볼별 CP 배열 대신 첫 CP 스칼라만 넘기는 대조군을 함께 돌려, 대조가 무엇을 잡아낼 수 있는지를 같은 표에 적는다 |
| 자원격자 자체 | 파일럿 좌표·가드밴드·DC 널은 규격서를 읽어 `src/waveforms.py` 에 세웠고, X410 캡처 대조는 실측 캠페인이 한다 |

### 재현

```bash
cd /home/yunjung/workspace/sionna2
~/.venvs/py312/bin/python src/viz_report2.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part08_illuminators.py
```

| | |
|---|---|
| 출력 | `outputs/report2_waveform_rcs.json` |
| 소요 | ① 3412 s [^68] · ② CPU 20초 안쪽 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| **절 1** «상시이면서 내용을 미리 아는 신호는 표준마다…» | 세 표준의 자원격자와 상시 기준신호 |

---


## 무엇을 채점하는가

격자를 신호로 바꾸는 **변조 단계**를 독립 구현으로 채점한다. 같은 자원격자를 Sionna PHY 의 `sionna.phy.ofdm.OFDMModulator` 에 넣고, 우리 변조기 출력과 상관·NMSE 를 잰다.

| 이 대조가 확인하는 것 | 무엇으로 |
|---|---|
| IFFT 규약 — fftshift 방향 · 정규화 | 두 구현의 시간파형 상관 |
| CP 복사와 심볼별 이어붙이기 순서 | 심볼별 CP 배열을 뺀 대조군과 비교 |
| 두 독립 구현의 시간파형 일치 | NMSE 바닥 |


![report03_f5_crosscheck](../outputs/figures/report03_f5_crosscheck.png)

**그림 1.** 같은 자원격자를 두 변조기에 넣으면 같은 시간파형이 나오는가?


## 채점 결과

| 표준 | 표본 수 | $f_s$ | 상관 | NMSE | CP 앞머리 | CP 배열을 뺀 대조군 |
|---|---|---|---|---|---|---|
| WiFi 802.11ac | 4160 [^69] | 80.00 MHz [^70] | 1.0000 [^57] | -138.3 dB [^60] | `[64]` | 1.0000 [^65] |
| LTE Rel-9 | 30720 [^71] | 30.72 MHz [^72] | 1.0000 [^58] | -135.6 dB [^61] | `[160, 144, 144, 144]` | 0.0634 [^63] |
| 5G NR Rel-16 | 61440 [^66] | 122.88 MHz [^67] | 1.0000 [^59] | -135.2 dB [^62] | `[352, 288, 288, 288]` | 0.0451 [^64] |


## 마지막 열이 대조의 분해력이다

두 구현이 같은 오해를 공유하면 대조가 통과해도 정보가 0 이다. 그 반론에 미리 답하려고 **일부러 틀린 대조군**을 같은 표에 싣는다.

재변조 쪽에 심볼별 CP 배열 대신 첫 CP 스칼라만 넘기면 두 번째 심볼부터 시간축이 어긋난다. CP 길이가 심볼마다 다른 LTE·5G 에서 상관이 무너지고, CP 가 심볼마다 같은 WiFi 는 그대로 1 이다 — 대조가 무엇을 잡아내는지가 그 열에 적혀 있다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| X410 으로 실제 셀을 캡처해 `src/waveforms.py` 의 격자와 대조한다 | CRS · SSB · VHT-LTF 의 격자 좌표가 실측으로 확정된다 | [리포트 14 절 4 «X410 의 12-bit ADC 동적범위가 직…»](14_robustness.ipynb) |
| 복조·등화 단계까지 같은 방식으로 채점한다 | 변조 밖 사슬의 독립 대조가 어디까지 서는지가 확정된다 | `src/waveforms_sionna.py` |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 16개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^57] | `outputs/report2_waveform_rcs.json` | `crosscheck.wifi.corr` | 1 |
| [^58] | `outputs/report2_waveform_rcs.json` | `crosscheck.lte.corr` | 1 |
| [^59] | `outputs/report2_waveform_rcs.json` | `crosscheck.nr.corr` | 1 |
| [^60] | `outputs/report2_waveform_rcs.json` | `crosscheck.wifi.nmse_db` | -138.3 |
| [^61] | `outputs/report2_waveform_rcs.json` | `crosscheck.lte.nmse_db` | -135.6 |
| [^62] | `outputs/report2_waveform_rcs.json` | `crosscheck.nr.nmse_db` | -135.2 |
| [^63] | `outputs/report2_waveform_rcs.json` | `crosscheck.lte.corr_bug` | 0.06338 |
| [^64] | `outputs/report2_waveform_rcs.json` | `crosscheck.nr.corr_bug` | 0.04507 |
| [^65] | `outputs/report2_waveform_rcs.json` | `crosscheck.wifi.corr_bug` | 1 |
| [^66] | `outputs/report2_waveform_rcs.json` | `crosscheck.nr.n` | 61440 |
| [^67] | `outputs/report2_waveform_rcs.json` | `crosscheck.nr.fs_mhz` | 122.9 |
| [^68] | `outputs/report2_waveform_rcs.json` | `meta.runtime_s` | 3412 |
| [^69] | `outputs/report2_waveform_rcs.json` | `crosscheck.wifi.n` | 4160 |
| [^70] | `outputs/report2_waveform_rcs.json` | `crosscheck.wifi.fs_mhz` | 80 |
| [^71] | `outputs/report2_waveform_rcs.json` | `crosscheck.lte.n` | 30720 |
| [^72] | `outputs/report2_waveform_rcs.json` | `crosscheck.lte.fs_mhz` | 30.72 |


---

## 절 6. 검출기가 실제로 쓰는 커널 그대로 모호함수를 그렸다



> ### 한 일
> **기준신호 하나가 거리-도플러 평면에 만드는 응답을 검출기와 같은 커널로 계산하고, 검출기의 거리도플러 출력과 대조해 두 값의 최대 편차를 쟀다.**

### 결과
1. 모호함수와 검출기 거리도플러 출력은 최대 0.144 dB [^73] 안에서 같다 (6 [^74]개 경우, −45 dB 이상 셀).
2. 거리 주엽은 $c/B_{ref}$ 예측의 89% [^75] ~ 94% [^76] 다(G1 세 파형).
3. 도플러 주엽은 여섯 경우 모두 $1/T_{CPI}$ 의 1.47 [^77]배 근처이고, 이 배수는 파형이 아니라 slow-time Hann 창이 정한다(`src/passive_process.py:142`).
4. 부엽과 ±PRF 레플리카는 표준마다 다르다 — 2D 부엽 최대가 LTE -5.3 [^78] · 5G -18.0 dB [^79] 이고, 레플리카는 WiFi -0.00 [^80] · LTE -23.27 dB [^81] 다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 커널 | 검출기가 쓰는 것과 **같은 커널**로 계산한다 — `benchmark/verify_ambiguity.py:150`, 검출기는 `src/passive_process.py:133` |
| 대조 방식 | 표준 × 점유 경우마다 −45 dB 이상 셀의 최대 편차를 재고 그 최대값을 싣는다 |
| 슬로타임 창 | 프레임과 프레임 사이 축에 Hann 창을 씌운다 — 도플러 주엽의 배수를 정하는 것이 이 창이다(`src/passive_process.py:142`) |
| 이 표의 PRF | **검출기 프레임률**이다. 물리 주기 기준의 접힘은 따로 잰다 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/verify_ambiguity.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part08_illuminators.py
```

| | |
|---|---|
| 출력 | `outputs/verify_ambiguity.json`, `outputs/report03_illuminators.json` |
| 소요 | ② GPU 1장 수 분 · ④ CPU 20초 안쪽 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| **절 4** «바이스태틱 거리 분해능은 c/B» | $\Delta R_b$ 와 잡음대역 규약 |

---


## 모호함수는 검출기의 눈이다

모호함수 $\chi(\tau, f_d)$ 는 기준신호 하나가 거리-도플러 평면에 만드는 응답이다. 표적이 점 하나여도 검출기 화면에는 이 모양이 찍힌다.

우리가 그리는 것은 **검출기가 쓰는 것과 같은 커널**이고, 검출기의 거리도플러 출력과 최대 0.144 dB [^73] (6 [^74]개 경우, −45 dB 이상 셀) 안에서 같다. 따로 계산한 그림이 아니라 **검출기 자신의 눈**이라는 뜻이다.


## 주엽 — 닫힌형과 대조

거리 주엽(응답에서 가장 높이 솟은 가운데 봉우리)은 $c/B_{ref}$ 예측의 89% [^75] ~ 94% [^76] 다(G1 세 파형).

도플러 주엽은 여섯 경우 모두 $1/T_{CPI}$ 의 1.47 [^77]배 근처이고, 이 배수는 파형이 아니라 **slow-time Hann 창**(프레임과 프레임 사이 축에 씌워 가장자리를 깎는 창)이 정한다.


![report03_f6_af_mainlobe](../outputs/figures/report03_f6_af_mainlobe.png)

**그림 1.** 측정한 모호함수 주엽이 닫힌형 예측과 몇 % 안에서 맞는가?


## 부엽과 도플러 레플리카

주엽 밖으로 새는 에너지는 두 가지로 나타난다. **부엽**은 강한 표적이 평면 다른 곳의 약한 표적을 덮는 정도이고, **±PRF 레플리카**는 무모호 속도를 넘은 표적이 되접혀 들어오는 세기다.

| 기준신호 | 2D 부엽 최대 | ±PRF 레플리카 | 프레임 내 시간점유 |
|---|---|---|---|
| WiFi VHT-LTF | -14.3 dB [^82] | -0.00 dB [^80] | 0.4% [^83] |
| LTE CRS | -5.3 dB [^78] | -23.27 dB [^81] | 42.9% [^84] |
| 5G SSB | -18.0 dB [^79] | -1.05 dB [^85] | 28.6% [^86] |


## 레플리카를 정하는 것은 점유율이 아니다

레플리카의 세기를 정하는 것은 **에너지가 프레임 안에 얼마나 퍼져 있는가**다. CRS 처럼 프레임 전체에 흩어지면 위상이 상쇄돼 레플리카가 죽고, LTF·SSB 처럼 앞쪽에 뭉치면 그대로 남는다.

이 표의 PRF 는 **검출기 프레임률**이다. 물리 주기 기준의 접힘은 **절 7** «5G SSB 는 걷는 드론에서 접힌다» 가 따로 잰다 — 그 편이 같은 표를 물리 반복률로 다시 세운다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| `benchmark/run_min_cell.py:74` 의 `frame_len()` 을 물리 SSB 주기(50 Hz [^87])로 확장한다 | 검출기 프레임률(2000 Hz [^88])과 40 [^89]배 벌어진 이 편의 표가 한 규약 위에 선다 | `benchmark/verify_ambiguity.py:108` |
| 부엽 최대를 표적 두 개가 있는 장면에서 다시 잰다 | 강한 표적이 약한 표적을 덮는 거리가 수치로 확정된다 | `benchmark/verify_ambiguity.py` |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 17개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^73] | `outputs/report03_illuminators.json` | `detector_af_max_err_db.value` | 0.1439 |
| [^74] | `outputs/report03_illuminators.json` | `detector_af_max_err_db.n_cases` | 6 |
| [^75] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.dR_ratio` | 0.895 |
| [^76] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.dR_ratio` | 0.9416 |
| [^77] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.dF_ratio` | 1.471 |
| [^78] | `outputs/verify_ambiguity.json` | `waveforms.lte_G1.psl_2d_db` | -5.309 |
| [^79] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.psl_2d_db` | -18 |
| [^80] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.doppler_replica_db` | -0.0002234 |
| [^81] | `outputs/verify_ambiguity.json` | `waveforms.lte_G1.doppler_replica_db` | -23.27 |
| [^82] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.psl_2d_db` | -14.33 |
| [^83] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.ref_time_duty` | 0.004 |
| [^84] | `outputs/verify_ambiguity.json` | `waveforms.lte_G1.ref_time_duty` | 0.4292 |
| [^85] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.doppler_replica_db` | -1.053 |
| [^86] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.ref_time_duty` | 0.2865 |
| [^87] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.physical.prf_physical_hz` | 50 |
| [^88] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.physical.prf_model_hz` | 2000 |
| [^89] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.physical.ratio` | 40 |


---

## 절 7. 5G SSB 는 걷는 드론에서 접힌다



> ### 한 일
> **세 기준신호의 **물리** 반복률에서 무모호 도플러를 계산하고, 기준 표적 속도의 참 도플러가 어디로 접히는지를 같은 표에 적었다.**

### 결과
1. SSB 의 물리 반복률은 50 Hz [^90] 이고 무모호 속도가 1.07 m/s [^91] 다 — 걷는 속도의 드론이 그 위에 있다.
2. 기준 표적 속도에서 참 도플러 64.0 Hz [^92] 가 14.0 Hz [^93] 로 접힌다.
3. 같은 조건에서 WiFi 는 무모호 속도 14.4 m/s [^94], LTE 는 40.7 m/s [^95] 로 참 도플러를 그대로 유지한다.
4. 접힘을 정하는 것은 물리 반복률 하나다 — CPI 는 도플러 가드 폭을 정하고, 모호속도는 표본화율의 성질이라 CPI 로 안 움직인다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 물리 반복률 | `TS 38.213` 의 기본 SSB 주기가 반복률을 고정한다. 검출기 프레임률과 다른 양이라 이름을 갈라 싣는다 |
| 접힘 계산 | 무모호 도플러 ±PRF/2 를 넘는 참 도플러를 그 구간으로 되접어 아래 표에 적는다 |
| 기준 표적 속도 | 이 프로젝트의 기준 표적 속도 하나에서 계산한다 — 속도를 바꾸면 접힌 자리도 바뀐다 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/verify_ambiguity.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part08_illuminators.py
```

| | |
|---|---|
| 출력 | `outputs/verify_ambiguity.json` |
| 소요 | ② GPU 1장 수 분 · ④ CPU 20초 안쪽 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| **절 2** «5G 는 좁고 드물다» | 5G 가 거리·속도 두 축에서 무는 대가 |
| **절 6** «검출기가 실제로 쓰는 커널 그대로 모호함수를…» | 검출기 커널이 만드는 응답의 모양 |

---


![report03_f7_af_sidelobe](../outputs/figures/report03_f7_af_sidelobe.png)

**그림 1.** 각 기준신호는 표적 에너지를 부엽과 도플러 레플리카에 얼마나 남기는가?


## 물리 반복률이 무모호 속도를 정한다

| 기준신호 | 물리 PRF | 무모호 속도 | 접히는가 |
|---|---|---|---|
| WiFi VHT-LTF | 1000 Hz [^96] | 14.4 m/s [^94] | 아니오 [^97] |
| LTE CRS | 1000 Hz [^98] | 40.7 m/s [^95] | 아니오 [^99] |
| 5G SSB | 50 Hz [^90] | 1.07 m/s [^91] | 예 [^100] |


## 접힌 자리는 어디인가

기준 표적 속도에서 5G 의 참 도플러 64.0 Hz [^92] 가 무모호 구간 ±25 Hz [^101] 안으로 되접혀 14.0 Hz [^93] 에 나타난다.

접힌 표적은 사라지는 것이 아니라 **엉뚱한 속도로 보고된다**. 그래서 이 대가는 감도가 아니라 판정의 정합성에 든다.


## 두 배의 대가의 나머지 절반

5G 는 좁아서 거리 눈금이 거칠고, 드물어서 속도 눈금이 접힌다 — **절 2** «5G 는 좁고 드물다 — 두 배의 대가를 치른다» 가 든 두 축의 뒤쪽이 여기다.

접힘을 정하는 것은 물리 반복률 50 Hz [^90] 하나이고, CPI 가 정하는 것은 도플러 가드 폭이다. 그 CPI 스윕은 [리포트 13 절 4 «CPI 를 늘리면 세 파형 모두 블라인드율이…»](13_results.ipynb) 가 싣고, CPI 로도 안 움직이는 잔여분은 [리포트 13 절 5 «모호속도는 표본화율의 성질이라 CPI 와 무관…»](13_results.ipynb) 가 든다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 검출기 CPI 24 ms [^102] 를 스윕해 SSB 도플러 가드 폭을 PRF 대비로 잰다 | 5G 상시 기준신호의 접힘이 단일 CPI 결과인지 체제인지가 수치로 갈린다 | `outputs/cpi_guard_sweep.json` → [리포트 13 절 4 «CPI 를 늘리면 세 파형 모두 블라인드율이…»](13_results.ipynb) |
| 표적 속도를 격자로 넓혀 접히는 속도 구간을 지도로 만든다 | 어느 속도대가 5G 에서 엉뚱한 속도로 보고되는지가 확정된다 | `benchmark/verify_ambiguity.py:108` |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 13개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^90] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.physical.prf_physical_hz` | 50 |
| [^91] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.physical.v_unamb_phys_ms` | 1.071 |
| [^92] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.physical.fd_true_hz` | 64.03 |
| [^93] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.physical.fd_aliased_phys_hz` | 14.03 |
| [^94] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.physical.v_unamb_phys_ms` | 14.39 |
| [^95] | `outputs/verify_ambiguity.json` | `waveforms.lte_G1.physical.v_unamb_phys_ms` | 40.67 |
| [^96] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.physical.prf_physical_hz` | 1000 |
| [^97] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.physical.aliased` | 아니오 |
| [^98] | `outputs/verify_ambiguity.json` | `waveforms.lte_G1.physical.prf_physical_hz` | 1000 |
| [^99] | `outputs/verify_ambiguity.json` | `waveforms.lte_G1.physical.aliased` | 아니오 |
| [^100] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.physical.aliased` | 예 |
| [^101] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.physical.fd_unamb_phys_hz` | 25 |
| [^102] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.physical.cpi_model_ms` | 24 |
